# Computer Security — Activity I: Hacking Password

Notebook นี้ทำตาม Exercise 1–6 จากเอกสาร **Ch03-Activity I-Hacking Password** โดยเน้นให้เห็นทั้งวิธีทดลองและเหตุผลเบื้องหลังผลลัพธ์

เอกสารกำหนดให้ศึกษา **dictionary attack, brute-force attack, rainbow-table attack, hashing และ salting** และให้เปรียบเทียบ MD5, SHA-1 และ bcrypt ในด้านความเร็วในการคำนวณ hash. fileciteturn0file0L1-L8

> **หมายเหตุด้านความปลอดภัย:** การทดลองนี้ใช้ hash และ dictionary ที่โจทย์ให้มาเพื่อการศึกษาเท่านั้น ไม่ควรนำไปใช้กับบัญชีหรือระบบที่ไม่มีสิทธิ์ทดสอบ

## 0. เตรียม environment

ติดตั้ง `bcrypt` ก่อน ถ้ายังไม่มี

```bash
pip install bcrypt
```

`hashlib` เป็น standard library ของ Python จึงไม่จำเป็นต้องติดตั้งแยกสำหรับ MD5/SHA-1

โจทย์ระบุ sample code สำหรับ SHA-1, MD5 และ bcrypt และใช้ `bcrypt.gensalt()` เพื่อสร้าง salt. fileciteturn0file0L9-L15

In [1]:
import hashlib
import bcrypt
import time
import math
import string
from pathlib import Path

# Exercise 1 — Dictionary attack + common substitutions

โจทย์ให้ SHA-1 target hash:

`d54cc1fe76f5186380a0939d2fc1723c44e8a5f7`

แนวคิดคือ:

1. อ่านคำจาก word list
2. สร้าง candidate password จากคำนั้น
3. ลอง substitution เช่น `o -> 0`, `l -> 1`, `i -> 1`
4. คำนวณ SHA-1
5. เปรียบเทียบกับ target hash

นี่เป็น **dictionary attack** เพราะเราไม่ได้ลองทุก string ที่เป็นไปได้ แต่เริ่มจากรายการคำที่มีโอกาสเป็น password สูงก่อน ตามโจทย์ของ Lab. fileciteturn0file0L17-L22

In [11]:
TARGET_SHA1 = "d54cc1fe76f5186380a0939d2fc1723c44e8a5f7"

def sha1_hex(text):
    return hashlib.sha1(text.encode("utf-8")).hexdigest()

from itertools import product

SUBSTITUTIONS = {
    "o": ["o", "0"],
    "l": ["l", "1"],
    "i": ["i", "1"],
}

def generate_candidates(word):
    word = word.strip()

    if not word:
        return set()

    # Convert to lowercase first
    word = word.lower()

    choices = []

    for char in word:

        # Possible substitution
        if char in SUBSTITUTIONS:
            substitution = SUBSTITUTIONS[char]
        else:
            substitution = [char]

        # For every substitution, allow lowercase and uppercase
        char_choices = set()

        for sub in substitution:
            char_choices.add(sub.lower())
            char_choices.add(sub.upper())

        choices.append(char_choices)

    # Generate ALL combinations
    candidates = set()

    for combination in product(*choices):
        candidate = "".join(combination)
        candidates.add(candidate)

    return candidates
# Quick demonstration
generate_candidates("thailand")

{'THA11AND',
 'THA11ANd',
 'THA11AnD',
 'THA11And',
 'THA11aND',
 'THA11aNd',
 'THA11anD',
 'THA11and',
 'THA1LAND',
 'THA1LANd',
 'THA1LAnD',
 'THA1LAnd',
 'THA1LaND',
 'THA1LaNd',
 'THA1LanD',
 'THA1Land',
 'THA1lAND',
 'THA1lANd',
 'THA1lAnD',
 'THA1lAnd',
 'THA1laND',
 'THA1laNd',
 'THA1lanD',
 'THA1land',
 'THAI1AND',
 'THAI1ANd',
 'THAI1AnD',
 'THAI1And',
 'THAI1aND',
 'THAI1aNd',
 'THAI1anD',
 'THAI1and',
 'THAILAND',
 'THAILANd',
 'THAILAnD',
 'THAILAnd',
 'THAILaND',
 'THAILaNd',
 'THAILanD',
 'THAILand',
 'THAIlAND',
 'THAIlANd',
 'THAIlAnD',
 'THAIlAnd',
 'THAIlaND',
 'THAIlaNd',
 'THAIlanD',
 'THAIland',
 'THAi1AND',
 'THAi1ANd',
 'THAi1AnD',
 'THAi1And',
 'THAi1aND',
 'THAi1aNd',
 'THAi1anD',
 'THAi1and',
 'THAiLAND',
 'THAiLANd',
 'THAiLAnD',
 'THAiLAnd',
 'THAiLaND',
 'THAiLaNd',
 'THAiLanD',
 'THAiLand',
 'THAilAND',
 'THAilANd',
 'THAilAnD',
 'THAilAnd',
 'THAilaND',
 'THAilaNd',
 'THAilanD',
 'THAiland',
 'THa11AND',
 'THa11ANd',
 'THa11AnD',
 'THa11And',
 'THa11aND',

In [4]:
# The exercise specifies the SecLists 10k common-password dictionary.
# If internet access is unavailable, replace WORDLIST_PATH with your own local copy.

WORDLIST_URL = (
    "https://raw.githubusercontent.com/danielmiessler/SecLists/master/"
    "Passwords/Common-Credentials/10k-most-common.txt"
)
WORDLIST_PATH = Path("10k-most-common.txt")

# Try to download the word list.
try:
    import urllib.request
    urllib.request.urlretrieve(WORDLIST_URL, WORDLIST_PATH)
    print(f"Downloaded word list to: {WORDLIST_PATH}")
except Exception as e:
    print("Could not download automatically.")
    print("Download the word list manually and place it beside this notebook.")
    print("Reason:", e)

Downloaded word list to: 10k-most-common.txt


In [12]:
def dictionary_attack(wordlist_path, target_hash):
    checked = 0

    with open(wordlist_path, "r", encoding="utf-8", errors="ignore") as f:
        for word in f:
            for candidate in generate_candidates(word):
                checked += 1
                if sha1_hex(candidate) == target_hash:
                    return candidate, checked

    return None, checked

if WORDLIST_PATH.exists():
    password, checked = dictionary_attack(WORDLIST_PATH, TARGET_SHA1)
    print("Candidate found:", password)
    print("Candidates checked:", checked)
else:
    print("Word list not found; skip this cell after placing the dictionary file.")

Candidate found: ThaiLanD
Candidates checked: 204388


### สิ่งที่ควรเข้าใจจาก Exercise 1

ถ้าโปรแกรมพบ password แปลว่า password นั้นอยู่ใน search space ที่เราสร้างจาก dictionary + substitutions

**จุดสำคัญ:** เราไม่ได้ "ถอด SHA-1" โดยตรง เพราะ hash function ถูกออกแบบให้ย้อนกลับไม่ได้ง่าย ๆ เรากำลังทำการ **guess → hash → compare** ซ้ำ ๆ

ดังนั้นความปลอดภัยของ password จึงขึ้นกับทั้ง:
- password มีความ predictable แค่ไหน
- attacker มี dictionary/ข้อมูลช่วยเดามากแค่ไหน
- hash function เร็วหรือช้าแค่ไหน

# Exercise 2 — เปรียบเทียบความเร็ว MD5, SHA-1 และ bcrypt

โจทย์ให้ทดลองว่าแต่ละ algorithm สามารถคำนวณ hash ได้กี่ครั้งในเวลาที่กำหนด และอย่างน้อยต้องมี MD5, SHA-1 และ bcrypt. fileciteturn0file0L24-L30

เราจะวัดเป็น **hashes per second (H/s)**

> หมายเหตุ: ตัวเลขขึ้นกับ CPU, Python version, OS และ bcrypt cost factor ดังนั้นให้ใช้ตัวเลขจากเครื่องของคุณเป็นผลการทดลองจริง

In [13]:
PASSWORD = b"Chulalongkorn"
BENCHMARK_SECONDS = 2.0

def benchmark_hash(function, duration=BENCHMARK_SECONDS):
    count = 0
    start = time.perf_counter()

    while time.perf_counter() - start < duration:
        function(PASSWORD)
        count += 1

    elapsed = time.perf_counter() - start
    return count, elapsed, count / elapsed

def md5_hash(data):
    return hashlib.md5(data).digest()

def sha1_hash(data):
    return hashlib.sha1(data).digest()

# bcrypt cost factor 12 is intentionally much slower than MD5/SHA-1.
BCRYPT_COST = 12
BCRYPT_SALT = bcrypt.gensalt(rounds=BCRYPT_COST)

def bcrypt_hash(data):
    return bcrypt.hashpw(data, BCRYPT_SALT)

results = {}

for name, func in [
    ("MD5", md5_hash),
    ("SHA-1", sha1_hash),
    ("bcrypt", bcrypt_hash),
]:
    count, elapsed, hps = benchmark_hash(func)
    results[name] = hps
    print(f"{name:8s}: {count:>8,d} hashes in {elapsed:.2f}s -> {hps:,.2f} H/s")

MD5     : 3,176,462 hashes in 2.00s -> 1,588,227.98 H/s
SHA-1   : 3,047,176 hashes in 2.00s -> 1,523,585.94 H/s
bcrypt  :        9 hashes in 2.10s -> 4.29 H/s


In [ ]:
# Optional: visualize the benchmark results as a simple table
import pandas as pd

benchmark_df = pd.DataFrame(
    [{"Algorithm": k, "Hashes_per_second": v} for k, v in results.items()]
).sort_values("Hashes_per_second", ascending=False)

benchmark_df

### ทำไม bcrypt ถึงช้ากว่า?

MD5 และ SHA-1 เป็น general-purpose hash functions ที่ออกแบบมาให้คำนวณเร็ว

bcrypt ถูกออกแบบมาเพื่อ **password hashing** จึงตั้งใจให้คำนวณช้ากว่า และมี **cost factor** ที่ปรับความยากได้

สำหรับ password cracking ความช้าเป็นข้อดี เพราะ attacker ต้องจ่ายเวลามากขึ้นสำหรับทุก password guess

ดังนั้นอย่าตีความว่า "bcrypt ช้ากว่า = algorithm แย่กว่า" ในบริบท password storage กลับเป็นคุณสมบัติที่ต้องการ

# Exercise 3 — ประมาณเวลาสำหรับ brute-force password

โจทย์กำหนดให้สมมติ password ใช้:
- uppercase
- lowercase
- numbers
- symbols

และให้ใช้ผลจาก Exercise 2 ประมาณเวลาที่ attacker ต้องใช้ในการ brute-force password. fileciteturn0file0L31-L35

ถ้ามี character set ขนาด `C` และ password ยาว `L`:

**จำนวน password ที่เป็นไปได้ = C^L**

ใน worst case attacker ต้องลองทั้งหมด `C^L` ค่า

เวลาประมาณ:

**time = C^L / hashes_per_second**

In [14]:
# Character set ตามโจทย์: upper + lower + numbers + symbols
# string.punctuation มี 32 symbols ใน Python
CHARSET_SIZE = len(string.ascii_uppercase + string.ascii_lowercase + string.digits + string.punctuation)

print("Character set size =", CHARSET_SIZE)

def brute_force_seconds(length, hashes_per_second, charset_size=CHARSET_SIZE):
    combinations = charset_size ** length
    return combinations / hashes_per_second

def format_seconds(seconds):
    units = [
        ("year", 365 * 24 * 3600),
        ("day", 24 * 3600),
        ("hour", 3600),
        ("minute", 60),
        ("second", 1),
    ]

    if seconds >= units[0][1]:
        return f"{seconds / units[0][1]:.3g} years"
    if seconds >= units[1][1]:
        return f"{seconds / units[1][1]:.3g} days"
    if seconds >= units[2][1]:
        return f"{seconds / units[2][1]:.3g} hours"
    if seconds >= units[3][1]:
        return f"{seconds / units[3][1]:.3g} minutes"
    return f"{seconds:.3g} seconds"

# Show estimates for several lengths using YOUR measured benchmark.
for algorithm, hps in results.items():
    print(f"\n{algorithm}")
    for length in [6, 8, 10, 12]:
        t = brute_force_seconds(length, hps)
        print(f"  length {length:2d}: {format_seconds(t)}")

Character set size = 94

MD5
  length  6: 5.03 days
  length  8: 122 years
  length 10: 1.08e+06 years
  length 12: 9.5e+09 years

SHA-1
  length  6: 5.24 days
  length  8: 127 years
  length 10: 1.12e+06 years
  length 12: 9.91e+09 years

bcrypt
  length  6: 5.1e+03 years
  length  8: 4.51e+07 years
  length 10: 3.98e+11 years
  length 12: 3.52e+15 years


### ระวังเรื่อง "1 ปี"

โจทย์ถามว่า password ที่เหมาะสมควรใช้เวลามากกว่า 1 ปีในการ brute-force หรือไม่. fileciteturn0file0L31-L35

แต่ในโลกจริง การกำหนด password policy ไม่ควรดูแค่ brute-force แบบ pure exhaustive search เพราะ attacker อาจใช้:
- dictionary attacks
- leaked passwords
- password reuse
- rules/substitutions
- probabilistic guesses

ดังนั้น **ความยาว + ความไม่ predictable + password hashing ที่เหมาะสม** สำคัญกว่าการพยายามทำให้ password มีทุกชนิดของ character เสมอไป

# Exercise 4 — bcrypt hash สามารถ brute-force ได้หรือไม่?

โจทย์ถามว่า ถ้า hash เป็น bcrypt จะทำ brute-force ได้จริงในทางปฏิบัติหรือไม่. fileciteturn0file0L37-L38

### คำตอบ

**ทำได้ในหลักการ แต่โดยทั่วไปแพงและช้ามากกว่า MD5/SHA-1**

เหตุผลคือ brute-force ไม่ได้หายไปเพียงเพราะใช้ bcrypt:

`guess password → bcrypt(guess) → compare`

ยังทำได้เหมือนเดิม

แต่ bcrypt ทำให้ **แต่ละ guess ใช้เวลามากขึ้น** ดังนั้นจำนวน guesses ต่อวินาทีลดลงอย่างมาก

จึงต้องพิจารณา:
- password length
- password predictability
- bcrypt cost factor
- computing resources ของ attacker

สรุป: **bcrypt ไม่ได้ทำให้ brute-force เป็นไปไม่ได้ แต่ทำให้การ brute-force มีต้นทุนสูงขึ้นมาก**

# Exercise 5 — Rainbow-table attack กับ bcrypt

โจทย์ถามว่า ถ้า hash เป็น bcrypt จะทำ rainbow-table attack ได้หรือไม่. fileciteturn0file0L40-L41

### คำตอบ

Rainbow table อาศัยการ precompute ความสัมพันธ์ระหว่าง password กับ hash เพื่อประหยัดเวลาตอนโจมตี

bcrypt ใช้ **salt ที่สุ่มต่อ password** ทำให้ password เดียวกันสามารถได้ hash ที่ต่างกันเมื่อ salt ต่างกัน

ตัวอย่าง:

In [15]:
password = b"Chulalongkorn"

h1 = bcrypt.hashpw(password, bcrypt.gensalt(rounds=12))
h2 = bcrypt.hashpw(password, bcrypt.gensalt(rounds=12))

print("Hash 1:", h1.decode())
print("Hash 2:", h2.decode())
print("Same password?", password == password)
print("Hashes equal?", h1 == h2)

Hash 1: $2b$12$NUGIJUcTGpWq/kv6D6ThpuKJ5qYmChPv1cY4c3sWH9/Hwvr88zOB2
Hash 2: $2b$12$6iBgXb.0FkP02S6Fk3CmU.a1dtIfdoKD5wJCPvn4IRGDn6q3130PW
Same password? True
Hashes equal? False


ผลที่ควรเห็นคือ **hash สองค่าไม่เหมือนกัน แม้ password จะเหมือนกัน**

นี่คือเหตุผลสำคัญที่ทำให้ rainbow table แบบ precomputed ทั่วไปไม่คุ้มค่าเมื่อใช้ salted password hashing เช่น bcrypt

อย่างไรก็ตาม attacker ยังสามารถทำ **online/offline guessing** กับ hash ที่ขโมยมาได้ เพียงแต่ต้องคำนวณ bcrypt ใหม่สำหรับแต่ละ guess

# Exercise 6 — ออกแบบการเก็บ password ใน database

โจทย์ให้พิจารณา:
- proper hash function
- salting
- cost factor
- database security. fileciteturn0file0L43-L45

## Strategy

1. **ไม่เก็บ plaintext password**
2. ใช้ password hashing algorithm ที่ออกแบบมาสำหรับ password เช่น bcrypt
3. ให้ algorithm สร้าง **random salt** ต่อ password
4. เลือก **cost factor** ให้สูงพอที่จะช้าอย่างเหมาะสมบน server แต่ไม่ทำให้ login ช้าเกินไป
5. เก็บ bcrypt hash ที่ได้ใน database
6. ตอน login ใช้ `bcrypt.checkpw()` เพื่อตรวจสอบ password
7. ป้องกัน database ด้วย access control, encryption และการจำกัดสิทธิ์ของ application account
8. ใช้ HTTPS/TLS ระหว่าง client กับ server
9. เพิ่ม rate limiting / account lockout สำหรับ online login attacks

In [16]:
# Example: secure password storage and verification with bcrypt

password = b"MyStrongExamplePassword123!"

# Registration:
stored_hash = bcrypt.hashpw(password, bcrypt.gensalt(rounds=12))

print("Store this value in DB (not the plaintext password):")
print(stored_hash.decode())

# Login:
login_attempt = b"MyStrongExamplePassword123!"

if bcrypt.checkpw(login_attempt, stored_hash):
    print("Login successful")
else:
    print("Invalid password")

Store this value in DB (not the plaintext password):
$2b$12$HjbqYDWYBajuJoLc0.TO5OBBc7uiMYj.0FFRsAx30n84x3dNegwWW
Login successful


## สรุปคำตอบทั้ง 6 ข้อ

| ข้อ | สิ่งที่เรียนรู้ | คำตอบหลัก |
|---|---|---|
| 1 | Dictionary attack | guess จาก word list + substitutions → hash → compare |
| 2 | Hash speed | MD5/SHA-1 เร็วกว่า bcrypt มาก; bcrypt ตั้งใจให้ช้ากว่า |
| 3 | Brute force | จำนวน possibilities = `C^L`; เวลา ≈ `C^L / H/s` |
| 4 | bcrypt brute force | ทำได้ในหลักการ แต่แต่ละ guess แพงและช้ากว่า |
| 5 | bcrypt rainbow table | random salt ทำให้ precomputed rainbow table ทั่วไปใช้ประโยชน์ได้น้อยมาก |
| 6 | Password storage | salted password hash + suitable cost factor + database/application security |

## สิ่งที่อาจารย์ต้องการให้เรา "เข้าใจ" จริง ๆ

ไม่ใช่แค่จำว่า **bcrypt ดีกว่า MD5/SHA-1**

แต่ควรเข้าใจ chain นี้:

**Weak password**
→ attacker เดาง่าย  
→ dictionary/brute-force มีประสิทธิภาพ  
→ ถ้า hash เร็ว เช่น MD5/SHA-1 ก็ลองได้จำนวนมาก  
→ ถ้าไม่มี salt ก็เสี่ยงต่อ precomputed/rainbow-table attacks

ในทางกลับกัน:

**Strong/unpredictable password**
+ **salted password hashing**
+ **appropriate cost factor**
+ **secure database**
→ ทำให้ password attack มีต้นทุนสูงขึ้นมาก